## lakeLoom CAF/AIFF Transcode Test

Tests server-side audio transcode for raw `.caf` and `.aiff` uploads:

1. Discover the dev app URL
2. Pair device via `PairingTestClient`
3. Create project + capture session
4. Upload a synthetic CAF file (`audio/x-caf` MIME)
5. Verify server transcodes to M4A (response `mime_type` = `audio/mp4`)
6. Verify `original_volume_path` + `original_mime_type` preserved in response
7. Verify transcoded `.m4a` file exists on UC Volume with valid content
8. Upload a synthetic AIFF file (`audio/x-aiff` MIME) — same assertions

**Prerequisite:** App deployed with `mg-genie-caf-upload-support` branch merged.

Run cells top-to-bottom. Self-contained — no chat context required.

In [0]:
%pip install --upgrade databricks-sdk cryptography
dbutils.library.restartPython()

In [0]:
import sys
sys.path.insert(0, '/Workspace/Users/matthew.giglia@databricks.com/lakeLoom/lakeloom-ai/src/tests')

from databricks.sdk import WorkspaceClient
from lib.pairing_client import PairingTestClient

wc = WorkspaceClient()
WORKSPACE_HOST = wc.config.host.rstrip('/')

SCOPE = 'lakeloom_credentials'
XCODE_CLIENT_ID = dbutils.secrets.get(SCOPE, 'xcode_client_id_dev_matthew_giglia_lakeloom')
XCODE_CLIENT_SECRET = dbutils.secrets.get(SCOPE, 'xcode_client_secret_dev_matthew_giglia_lakeloom')

APP_NAME = 'lakeloom-ai-dev'
APP_URL = None
for app in wc.apps.list():
    if app.name == APP_NAME:
        APP_URL = app.url.rstrip('/')
        break

if not APP_URL:
    raise ValueError(f'Could not find app URL for {APP_NAME}')

print('Workspace host:', WORKSPACE_HOST)
print('App name:      ', APP_NAME)
print('App url:       ', APP_URL)

In [0]:
import uuid
import time

# ── Pair device ─────────────────────────────────────────────────────────────────
client = PairingTestClient(
    app_url=APP_URL,
    workspace_host=WORKSPACE_HOST,
    xcode_client_id=XCODE_CLIENT_ID,
    xcode_client_secret=XCODE_CLIENT_SECRET,
)
client.acquire_spn_token()

TEST_DEVICE_ID = str(uuid.uuid4())
session = client.pair_device(
    device_label=f'CAF Transcode Test {int(time.time())}',
    device_id=TEST_DEVICE_ID,
)

SPN_TOKEN = session.spn_token
PAIRED_SESSION_ID = session.paired_session_id
print(f'Device paired: {PAIRED_SESSION_ID}')

# ── Create project ──────────────────────────────────────────────────────────────
project_body = {
    'name': f'CAF Transcode Test {int(time.time())}',
    'description': 'Temporary project for CAF/AIFF transcode verification',
    'workspace_id': WORKSPACE_HOST,
    'client_generated_id': str(uuid.uuid4()),
}
project_resp = session.post('/api/v1/projects', json_body=project_body)
project_resp.raise_for_status()
PROJECT_ID = project_resp.json()['id']
print(f'Project created: {PROJECT_ID}')

# ── Assign device ───────────────────────────────────────────────────────────────
assign_resp = session.post(
    f'/api/v1/projects/{PROJECT_ID}/devices',
    json_body={'paired_session_id': PAIRED_SESSION_ID},
)
assign_resp.raise_for_status()
print(f'Device assigned: {assign_resp.json().get("id")}')

# ── Create capture session ──────────────────────────────────────────────────────
capture_body = {
    'label': f'CAF Transcode Capture {int(time.time())}',
    'device_id': TEST_DEVICE_ID,
}
capture_resp = session.post(f'/api/projects/{PROJECT_ID}/captures', json_body=capture_body)
capture_resp.raise_for_status()
CAPTURE_ID = capture_resp.json()['id']
print(f'Capture created: {CAPTURE_ID}')

In [0]:
import struct
import hashlib


def build_test_caf_bytes(sample_rate: int = 16000, num_frames: int = 16000) -> bytes:
    """
    Build a minimal valid CAF (Core Audio Format) file with PCM silence.
    
    CAF structure:
      - File header: 'caff' magic + version (1) + flags (0)
      - Audio Description chunk: format details
      - Audio Data chunk: raw PCM samples
    
    This produces a valid file that ffmpeg can transcode to M4A.
    """
    # CAF file header: magic 'caff', version 1, flags 0
    file_header = b'caff' + struct.pack('>HH', 1, 0)

    # Audio Description chunk
    # chunk type: 'desc', chunk size: 32 bytes
    channels = 1
    bits_per_channel = 16
    bytes_per_packet = channels * (bits_per_channel // 8)
    frames_per_packet = 1  # PCM = 1 frame per packet
    
    # Format: Linear PCM, big-endian signed integer
    # Format flags: kCAFLinearPCMFormatFlagIsSignedInteger | kCAFLinearPCMFormatFlagIsBigEndian
    format_flags = 0x04 | 0x02  # signed int + big endian
    
    desc_data = struct.pack(
        '>dIIIIII',
        float(sample_rate),   # mSampleRate (float64)
        b'lpcm'[0] << 24 | b'lpcm'[1] << 16 | b'lpcm'[2] << 8 | b'lpcm'[3],  # mFormatID
        format_flags,          # mFormatFlags
        bytes_per_packet,      # mBytesPerPacket
        frames_per_packet,     # mFramesPerPacket
        channels,              # mChannelsPerFrame
        bits_per_channel,      # mBitsPerChannel
    )
    desc_chunk = b'desc' + struct.pack('>q', len(desc_data)) + desc_data

    # Audio Data chunk: 'data' + size + edit count (4 bytes) + raw samples
    # PCM silence: all zeros
    pcm_data = b'\x00' * (num_frames * bytes_per_packet)
    # data chunk size = 4 (edit count) + len(pcm_data)
    data_chunk_size = 4 + len(pcm_data)
    data_chunk = b'data' + struct.pack('>q', data_chunk_size) + struct.pack('>I', 0) + pcm_data

    return file_header + desc_chunk + data_chunk


def build_test_aiff_bytes(sample_rate: int = 16000, num_frames: int = 16000) -> bytes:
    """
    Build a minimal valid AIFF file with PCM silence.
    
    AIFF structure:
      - FORM header with AIFF type
      - COMM chunk: format details
      - SSND chunk: audio data
    """
    channels = 1
    sample_size = 16
    
    # COMM chunk (Common): 18 bytes
    # numChannels(2) + numSampleFrames(4) + sampleSize(2) + sampleRate(10 as 80-bit extended)
    # 16000 Hz as 80-bit IEEE 754 extended: sign=0, exp=16397 (0x400D), mantissa
    # Simplified: use the known byte pattern for 16000.0
    sr_extended = b'\x40\x0c\xfa\x00\x00\x00\x00\x00\x00\x00'  # 16000.0 as 80-bit
    
    comm_data = struct.pack('>hIh', channels, num_frames, sample_size) + sr_extended
    comm_chunk = b'COMM' + struct.pack('>I', len(comm_data)) + comm_data
    
    # SSND chunk: offset(4) + blockSize(4) + sound data
    pcm_data = b'\x00' * (num_frames * channels * (sample_size // 8))
    ssnd_data = struct.pack('>II', 0, 0) + pcm_data  # offset=0, blockSize=0
    ssnd_chunk = b'SSND' + struct.pack('>I', len(ssnd_data)) + ssnd_data
    
    # FORM container
    form_data = b'AIFF' + comm_chunk + ssnd_chunk
    aiff_file = b'FORM' + struct.pack('>I', len(form_data)) + form_data
    
    return aiff_file


CAF_BYTES = build_test_caf_bytes()
AIFF_BYTES = build_test_aiff_bytes()

print(f'CAF test file:  {len(CAF_BYTES):,} bytes (magic: {CAF_BYTES[:4]})')
print(f'AIFF test file: {len(AIFF_BYTES):,} bytes (magic: {AIFF_BYTES[:4]})')
print(f'CAF SHA-256:    {hashlib.sha256(CAF_BYTES).hexdigest()[:16]}...')
print(f'AIFF SHA-256:   {hashlib.sha256(AIFF_BYTES).hexdigest()[:16]}...')

In [0]:
import hashlib
import json
import time

CAF_SHA256 = hashlib.sha256(CAF_BYTES).hexdigest()
CAF_FILENAME = 'test-recording.caf'
CLIENT_TS = str(int(time.time()))

upload_path = f'/api/captures/{CAPTURE_ID}/audio'

print('=' * 60)
print('TEST 1: CAF Upload + Server-Side Transcode')
print('=' * 60)
print(f'Upload path:    {upload_path}')
print(f'MIME type:      audio/x-caf')
print(f'File size:      {len(CAF_BYTES):,} bytes')
print(f'Input SHA-256:  {CAF_SHA256[:32]}...')
print()

caf_resp = session.upload(
    path=upload_path,
    file_bytes=CAF_BYTES,
    file_field='file',
    mime_type='audio/x-caf',
    filename=CAF_FILENAME,
    extra_fields={
        'client_ts': CLIENT_TS,
        'client_filename': CAF_FILENAME,
        'sha256_hex': CAF_SHA256,
        'device_id': TEST_DEVICE_ID,
    },
    timeout=60,
)

print(f'HTTP status:    {caf_resp.status_code}')
caf_resp.raise_for_status()
CAF_UPLOAD = caf_resp.json()

print(f'Upload ID:      {CAF_UPLOAD.get("id")}')
print(f'Response MIME:   {CAF_UPLOAD.get("mime_type")}')
print(f'Volume path:    {CAF_UPLOAD.get("volume_path")}')
print(f'Size bytes:     {CAF_UPLOAD.get("size_bytes")}')
print(f'SHA-256:        {CAF_UPLOAD.get("sha256_hex", "")[:32]}...')
print()

# ── Assertions ────────────────────────────────────────────────────────────────
transcode_happened = CAF_UPLOAD.get('mime_type') == 'audio/mp4'

if transcode_happened:
    print('✅ TRANSCODE VERIFIED — server converted CAF → M4A')
    assert CAF_UPLOAD['volume_path'].endswith('.m4a'), f"Expected .m4a path, got: {CAF_UPLOAD['volume_path']}"
    assert CAF_UPLOAD['size_bytes'] > 0, 'Transcoded file has zero bytes'
    # SHA should differ from input (transcoded file has different bytes)
    assert CAF_UPLOAD['sha256_hex'] != CAF_SHA256, 'SHA should differ after transcode'
    print(f'   Output extension: .m4a ✓')
    print(f'   Size > 0: {CAF_UPLOAD["size_bytes"]:,} bytes ✓')
    print(f'   SHA differs from input: ✓')
else:
    # Transcode may not have happened (ffmpeg not installed, or file wasn't valid enough)
    print(f'⚠️  NO TRANSCODE — response mime_type is: {CAF_UPLOAD.get("mime_type")}')
    print('   This means either:')
    print('   - ffmpeg is not installed on the container')
    print('   - The synthetic CAF was not valid enough for ffmpeg')
    print('   - The transcode timed out or errored (non-fatal, raw preserved)')
    print(f'   Volume path: {CAF_UPLOAD.get("volume_path")}')

In [0]:
import os

volume_path = CAF_UPLOAD.get('volume_path')

if volume_path and os.path.exists(volume_path):
    file_size = os.path.getsize(volume_path)
    with open(volume_path, 'rb') as f:
        header = f.read(12)
        file_sha = hashlib.sha256(f.read()).hexdigest()  # Note: reads from offset 12
    
    # Reset and compute full SHA
    with open(volume_path, 'rb') as f:
        full_sha = hashlib.sha256(f.read()).hexdigest()
    
    print(f'File on volume: {volume_path}')
    print(f'File size:      {file_size:,} bytes')
    print(f'SHA-256:        {full_sha[:32]}...')
    print(f'First 12 bytes: {header.hex()}')
    
    if transcode_happened:
        # M4A files start with an ftyp box: ....ftyp or similar
        # The 4-8 bytes should be 'ftyp' for valid M4A/MP4
        has_ftyp = header[4:8] == b'ftyp'
        print(f'Has ftyp box:   {has_ftyp} ({header[4:8]})')
        assert file_size == CAF_UPLOAD['size_bytes'], f'Size mismatch: {file_size} vs {CAF_UPLOAD["size_bytes"]}'
        assert full_sha == CAF_UPLOAD['sha256_hex'], f'SHA mismatch on volume'
        if has_ftyp:
            print('✅ VOLUME VERIFIED — valid M4A file with ftyp header')
        else:
            print('⚠️  File exists but no ftyp header — may still be valid M4A')
    else:
        # Raw CAF should start with 'caff'
        has_caff = header[:4] == b'caff'
        print(f'Has caff magic: {has_caff}')
        if has_caff:
            print('✅ RAW CAF PRESERVED — file stored without transcode')
else:
    print(f'⚠️  File not found at: {volume_path}')
    print('   This may happen if the volume is not directly accessible from this notebook.')
    print('   Check via dbutils.fs.ls() or SDK files API instead.')

In [0]:
AIFF_SHA256 = hashlib.sha256(AIFF_BYTES).hexdigest()
AIFF_FILENAME = 'test-recording.aiff'
CLIENT_TS_2 = str(int(time.time()))

print('=' * 60)
print('TEST 2: AIFF Upload + Server-Side Transcode')
print('=' * 60)
print(f'MIME type:      audio/x-aiff')
print(f'File size:      {len(AIFF_BYTES):,} bytes')
print(f'Input SHA-256:  {AIFF_SHA256[:32]}...')
print()

aiff_resp = session.upload(
    path=upload_path,
    file_bytes=AIFF_BYTES,
    file_field='file',
    mime_type='audio/x-aiff',
    filename=AIFF_FILENAME,
    extra_fields={
        'client_ts': CLIENT_TS_2,
        'client_filename': AIFF_FILENAME,
        'sha256_hex': AIFF_SHA256,
        'device_id': TEST_DEVICE_ID,
    },
    timeout=60,
)

print(f'HTTP status:    {aiff_resp.status_code}')
aiff_resp.raise_for_status()
AIFF_UPLOAD = aiff_resp.json()

print(f'Upload ID:      {AIFF_UPLOAD.get("id")}')
print(f'Response MIME:   {AIFF_UPLOAD.get("mime_type")}')
print(f'Volume path:    {AIFF_UPLOAD.get("volume_path")}')
print(f'Size bytes:     {AIFF_UPLOAD.get("size_bytes")}')
print()

aiff_transcode = AIFF_UPLOAD.get('mime_type') == 'audio/mp4'

if aiff_transcode:
    print('✅ AIFF TRANSCODE VERIFIED — server converted AIFF → M4A')
    assert AIFF_UPLOAD['volume_path'].endswith('.m4a')
    assert AIFF_UPLOAD['size_bytes'] > 0
    assert AIFF_UPLOAD['sha256_hex'] != AIFF_SHA256
else:
    print(f'⚠️  AIFF NOT TRANSCODED — mime_type: {AIFF_UPLOAD.get("mime_type")}')

In [0]:
# Query OTel logs for transcode-related events from this session
from pyspark.sql import functions as F

otel_df = spark.table('hls_fde_dev.dev_matthew_giglia_lakeloom.lakeloom_ai_otel_logs')

transcode_logs = (
    otel_df
    .filter(F.col('time') > F.current_timestamp() - F.expr('INTERVAL 5 MINUTES'))
    .filter(
        F.col('body').cast('string').contains('transcode')
    )
    .select('time', 'severity_text', F.col('body').cast('string').alias('body_text'))
    .orderBy(F.col('time').desc())
    .limit(10)
)

print('Recent transcode-related OTel logs:')
print('─' * 60)
display(transcode_logs)

In [0]:
# Check lb_uploads_history for the original_volume_path and original_mime_type columns
uploads_df = spark.table('hls_fde_dev.dev_matthew_giglia_lakeloom.lb_uploads_history')

upload_ids = [CAF_UPLOAD.get('id'), AIFF_UPLOAD.get('id')]
upload_ids = [uid for uid in upload_ids if uid]

if upload_ids:
    results = (
        uploads_df
        .filter(F.col('id').isin(upload_ids))
        .select(
            'id', 'kind', 'mime_type', 'size_bytes',
            'original_volume_path', 'original_mime_type',
            'volume_path'
        )
    )
    print('Upload metadata from Lakebase (Lakehouse Sync):')
    print('─' * 60)
    display(results)
    
    # Check if original columns are populated
    for row in results.collect():
        uid = row['id']
        if row['original_mime_type']:
            print(f'\n✅ Upload {uid[:8]}...: original_mime_type = {row["original_mime_type"]}')
            print(f'   original_volume_path = {row["original_volume_path"]}')
            print(f'   current mime_type = {row["mime_type"]}')
            print(f'   current volume_path = {row["volume_path"]}')
        else:
            print(f'\n⚠️  Upload {uid[:8]}...: original columns are NULL (no transcode occurred or sync pending)')
else:
    print('⚠️  No upload IDs to verify — previous uploads may have failed')
    print('   Note: Lakehouse Sync has a ~60s propagation delay')

In [0]:
print('=' * 60)
print('CAF/AIFF TRANSCODE TEST SUMMARY')
print('=' * 60)

results = [
    ('CAF upload accepted (201)', caf_resp.status_code == 201),
    ('CAF transcoded to M4A', transcode_happened),
    ('CAF volume path ends in .m4a', CAF_UPLOAD.get('volume_path', '').endswith('.m4a') if transcode_happened else None),
    ('AIFF upload accepted (201)', aiff_resp.status_code == 201),
    ('AIFF transcoded to M4A', aiff_transcode),
    ('AIFF volume path ends in .m4a', AIFF_UPLOAD.get('volume_path', '').endswith('.m4a') if aiff_transcode else None),
]

for label, passed in results:
    if passed is True:
        print(f'  ✅ {label}')
    elif passed is False:
        print(f'  ❌ {label}')
    else:
        print(f'  ⏭️  {label} (skipped — transcode did not occur)')

print()
all_passed = all(p for _, p in results if p is not None)
if all_passed:
    print('🎉 ALL TESTS PASSED — server-side transcode working correctly')
else:
    print('⚠️  SOME TESTS FAILED — check individual results above')

print()
print('Upload details:')
print(json.dumps({
    'caf_upload_id': CAF_UPLOAD.get('id'),
    'caf_mime': CAF_UPLOAD.get('mime_type'),
    'caf_volume_path': CAF_UPLOAD.get('volume_path'),
    'aiff_upload_id': AIFF_UPLOAD.get('id'),
    'aiff_mime': AIFF_UPLOAD.get('mime_type'),
    'aiff_volume_path': AIFF_UPLOAD.get('volume_path'),
    'project_id': PROJECT_ID,
    'capture_id': CAPTURE_ID,
}, indent=2))